# OPUS‑MT Fine‑Tuning & Minerva Refinement

In [ ]:

# OPUS‑MT FINE‑TUNING + MINERVA REFINEMENT
# --------------------------------------------------
!pip install --no-deps xformers trl peft accelerate bitsandbytes

import pandas as pd, torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)

# --- Dataset ---
df = pd.read_excel("/content/drive/MyDrive/dataset_augmented(without_last10_for_testing).xlsx")[["Sentence", "TRADUZIONE ITALIANA"]]
raw_ds = Dataset.from_pandas(df)
split = raw_ds.train_test_split(test_size=0.1)
train_ds, test_ds = split["train"], split["test"]

model_ckpt = "Helsinki-NLP/opus-mt-en-it"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

def preprocess(batch):
    src = tokenizer(batch["Sentence"], padding="max_length", truncation=True, max_length=128)
    with tokenizer.as_target_tokenizer():
        tgt = tokenizer(batch["TRADUZIONE ITALIANA"], padding="max_length", truncation=True, max_length=128)
    src["labels"] = tgt["input_ids"]
    return src

train_ds = train_ds.map(preprocess, batched=True, remove_columns=train_ds.column_names)
test_ds  = test_ds.map(preprocess,  batched=True, remove_columns=test_ds.column_names)

model = AutoModelForSeq2SeqLM.from_pretrained(model_ckpt, device_map="auto")

args = Seq2SeqTrainingArguments(
    output_dir="./results_opus",
    per_device_train_batch_size=4,
    num_train_epochs=30,
    learning_rate=2e-5,
    logging_steps=50,
    save_total_limit=3,
    predict_with_generate=True,
    report_to="none",
)

trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=test_ds, tokenizer=tokenizer)
print("▶ Fine‑tuning OPUS‑MT ...")
trainer.train()
print("✅ Done!")

# --- Inference ---
sample = "sia in mezzo tra me e te: con noi non puo' tu già più lungamente dimorare, ch'io non lo sofferrò e non lo lascerò."
tokenised = tokenizer(sample, return_tensors="pt").to(model.device)
with torch.no_grad():
    generated = model.generate(**tokenised, max_length=128, num_beams=4, early_stopping=True)
opus_translation = tokenizer.decode(generated[0], skip_special_tokens=True)
print("OPUS →", opus_translation)

# --- Refinement with Minerva‑3B ---
from transformers import pipeline
minerva_id = "sapienzanlp/Minerva-3B-base-v1.0"
refiner = pipeline(
    "text-generation",
    model=minerva_id,
    device_map="auto",
    model_kwargs={"torch_dtype": torch.bfloat16},
)

refine_prompt = f'''
Rifinisci questa traduzione grezza da italiano arcaico (dopo il marcatore ARCAICO:) a italiano moderno (dopo il marcatore INPUT:).
Produci solo una frase rifinita in italiano moderno.

ARCAICO: {sample}
INPUT:   {opus_translation}
OUTPUT:'''
refined = refiner(refine_prompt, max_new_tokens=128, do_sample=False)[0]["generated_text"].split("OUTPUT:")[-1].strip()
print("Minerva →", refined)
